# Setup

In [1]:
# Base class
from src.Base.Config import Config

# Project class
from src.BetaMouv.Trajectory import TrajectoryDLC, DLCTracking
import src.BetaMouv.Behavior as Behavior
import src.utils as u

# other lib
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
cfg = Config(Path("./config.yaml"))
project = cfg.load_project()

custom_params = {"axes.spines.right": False, "axes.spines.top": False}
sns.set_theme("talk", style="ticks", rc=custom_params, palette="pastel")

analysis_dir = u.make_dir(project.paths.analysis(project.trialgroup.keep_val))
df_dir = u.make_dir(analysis_dir / "trials_df")

project.trialgroup.crop_laser_period = True
project.trialgroup.remove_NOstim = True

log: config unchanged (config_20261102_152511.yaml)

 CONTRA_CHR_#517_#525_#521_#531


### Load joblib trials
can take a bit of time

In [3]:
project.trialgroup.load_trials()

Loading joblib: 100%|██████████| 228/228 [00:02<00:00, 105.14it/s]

--> 59/228 FILES LOADED
--> 4933 trials


In [4]:
BODYPARTS = [
    "Hand_base",
    "Base_finger_2",
    "Base_finger_3",
    "Phalanx_1_finger_2",
    "Phalanx_1_finger_3",
    "finger_1",
    "finger_2",
    "finger_3",
    "soft_pad",
    "Front_PAD_corner",
    "Back_PAD_corner",
    "Lever_ball",
    "Lever_base",
    "Lever_corner",
    "hand", 
    "wrist",
]
preprocess_bodyparts = [
    "Hand_base",
    "Base_finger_2",
    "Base_finger_3",
    "Phalanx_1_finger_2",
    "Phalanx_1_finger_3",
    "finger_1",
    "finger_2",
    "finger_3",
    "soft_pad",
    "Front_PAD_corner",
    "Back_PAD_corner",
    "Lever_ball",
    "Lever_base",
    "Lever_corner",
]
skeleton=[
    ("Hand_base", "Base_finger_2"),
    ("Hand_base", "Base_finger_3"),
    ("Hand_base", "soft_pad"),
    ("Base_finger_2", "Phalanx_1_finger_2"),
    ("Base_finger_3", "Phalanx_1_finger_3"),
    ("Phalanx_1_finger_2", "finger_2"),
    ("Phalanx_1_finger_3", "finger_3"),
    ("Front_PAD_corner", "Back_PAD_corner"),
    ("Lever_base", "Lever_corner"),
]
color_map = {
    "Hand_base": "lightskyblue",
    "Base_finger_2": "lightskyblue",
    "Base_finger_3": "lightskyblue",
    "Phalanx_1_finger_2": "lightskyblue",
    "Phalanx_1_finger_3": "lightskyblue",
    "finger_1": "lightskyblue",
    "finger_2": "lightskyblue",
    "finger_3": "lightskyblue",
    "soft_pad": "lightskyblue",
    "Front_PAD_corner": "olive",
    "Back_PAD_corner": "olive",
    "Lever_ball": "tomato",
    "Lever_base": "tomato",
    "Lever_corner": "tomato",
    "hand": "lightskyblue", 
    "wrist": "lightskyblue",
}
output_dir = project.paths.data_root / "keypoint_moseq" / "dlc_data"
output_dir.mkdir(parents=True, exist_ok=True)

In [5]:
print(BODYPARTS)
print(len(BODYPARTS))

['Hand_base', 'Base_finger_2', 'Base_finger_3', 'Phalanx_1_finger_2', 'Phalanx_1_finger_3', 'finger_1', 'finger_2', 'finger_3', 'soft_pad', 'Front_PAD_corner', 'Back_PAD_corner', 'Lever_ball', 'Lever_base', 'Lever_corner', 'hand', 'wrist']
16


In [ ]:
n_trial = 200
n=0

for trial in tqdm(project.trialgroup.trials, desc="cropping"):
    if n>n_trial: 
        break
    if not trial.is_valid() or trial.date.month == 5 or any(stat["is_valid"] == False for bp, stat in trial.trajectories.items()):
        continue

    save_as: Path = output_dir / f"{trial.name}.csv"
    # if save_as.exists(): 
    #     continue

    raw_tracking = DLCTracking(path=trial.pred_path,
                                view=trial.camera_view, 
                                bodyparts=BODYPARTS,
                                time_pad_off=trial.time_pad_off,
                                cm_per_pixel=trial.cm_per_pixel,
                                shift=trial.camera_shift)

    clean_tracking = DLCTracking(path=trial.clean_pred_path,
                                view=trial.camera_view, 
                                bodyparts=BODYPARTS,
                                time_pad_off=trial.time_pad_off,
                                cm_per_pixel=trial.cm_per_pixel,
                                shift=trial.camera_shift, 
                                status=trial.trajectories)

    raw_coords_set = raw_tracking.load(False)
    clean_coords_set = clean_tracking.load()

    for bp in BODYPARTS: 
        if bp not in clean_coords_set.keys(): 
            clean_coords_set[bp] = raw_coords_set[bp]
            clean_coords_set[bp].coords = clean_coords_set[bp].crop_xy(start=trial.time_pad_off - 0.1, end=trial.time_pad_off + 0.4)
            continue

        traj = clean_coords_set[bp] 
        traj.coords = traj.crop_xy(start=trial.time_pad_off - 0.1, end=trial.time_pad_off + 0.4)
        
        # since it in cm, put back in pixel, without removing the shift
        traj.coords["x"] = traj.coords["x"] / trial.cm_per_pixel
        traj.coords["y"] = traj.coords["y"] / trial.cm_per_pixel

    # format saving
    data = {bp: traj.coords.set_index("t")
        for bp, traj in clean_coords_set.items()}

    df = pd.concat({raw_tracking.scorer: pd.concat(data, axis=1)}, axis=1, names=["scorer", "bodyparts", "coords"])
    if  len(df) != 63: 
        continue

    # Replace index with frame_num (0, 1, ..., len(df)-1)
    df.index = pd.RangeIndex(start=0, stop=len(df), name="frame_num")

    df.to_csv(output_dir / f"{trial.name}.csv")
    trial.save_yaml(output_dir / f"{trial.name}.yaml")
    n+=1



In [8]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_tracking(df, skeleton, indices=None, delta_T=10, by_label=False, color_by_time=True) :
    bodyparts = bodyparts_to_use

    if indices is None:
        sampled = df.iloc[::delta_T]
        title = f"Bodypart tracking (every {delta_T} frames)"
    else:
        sampled = df.loc[indices] if by_label else df.iloc[indices]
        title = f"Bodypart tracking ({len(sampled)} selected frames)"
    
    n = len(sampled)
    t_norm = np.linspace(0.1, 1.0, n)  # 0.3 -> skip the near-white end of the scale
    colors = px.colors.qualitative.Plotly

    fig = go.Figure()

    # --- Skeleton ---
    if color_by_time:
        # One trace per sampled frame, colored along a sequential scale
        n = len(sampled)
        frame_colors = px.colors.sample_colorscale(
            "Blues", [i / max(n - 1, 1) for i in range(n)])

        for k, (idx, row) in enumerate(sampled.iterrows()):
            xs, ys = [], []
            for a, b in skeleton:
                xs += [row[a]["x"], row[b]["x"], None]
                ys += [row[a]["y"], row[b]["y"], None]
            fig.add_trace(go.Scatter(
                x=xs, y=ys, mode="lines",
                line=dict(width=1.5, color=frame_colors),
                opacity=0.8,
                hoverinfo="skip",
                name=f"t={idx}",
                legendgroup="skeleton",
                showlegend=False,
            ))

    else:
        for a, b in skeleton:
            seg_colors = px.colors.sample_colorscale(color_scale_map[a], t_norm)

            xa, xb = sampled[a]["x"].to_numpy(), sampled[b]["x"].to_numpy()
            ya, yb = sampled[a]["y"].to_numpy(), sampled[b]["y"].to_numpy()

            for k in range(n):
                fig.add_trace(go.Scatter(
                    x=[xa[k], xb[k]],
                    y=[ya[k], yb[k]],
                    mode="lines",
                    line=dict(width=5, color=seg_colors[k]),
                    opacity=1,
                    hoverinfo="skip",
                    showlegend=False,
                ))

    # --- Points (drawn after, so they sit on top) ---
    for i, bp in enumerate(bodyparts):
        point_colors = px.colors.sample_colorscale(color_scale_map[bp], t_norm)
        fig.add_trace(go.Scatter(
            x=sampled[bp]["x"], y=sampled[bp]["y"],
            mode="markers",
            name=bp,
            marker=dict(size=10, color=point_colors, opacity=0.9,
                        line=dict(width=0.5, color="black")),
        ))

    fig.update_layout(
        title=title,
        xaxis_title="X", yaxis_title="Y",
        template="plotly_white", width=900, height=800,
    )
    fig.update_yaxes(
        # autorange="reversed", 
                     scaleanchor="x")
    fig.write_html(u.make_path(output_dir / "figs", f"{path.stem}" ))
    # fig.savefig(u.make_path(output_dir / "figs", f"{path.stem}.png" ))
    
    return fig

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
bodyparts_to_use = [
    "Hand_base",
    "Base_finger_2",
    "Base_finger_3",
    "Phalanx_1_finger_2",
    "Phalanx_1_finger_3",
    "finger_1",
    "finger_2",
    "finger_3",
    "soft_pad",
    "Front_PAD_corner",
    "Back_PAD_corner",
    "Lever_ball",
    "Lever_base",
    "Lever_corner",
    "hand", 
    "wrist"
]
color_scale_map = {
    "Hand_base": "Blues",
    "Base_finger_2": "Blues",
    "Base_finger_3": "Blues",
    "Phalanx_1_finger_2": "Blues",
    "Phalanx_1_finger_3": "Blues",
    "finger_1": "Blues",
    "finger_2": "Blues",
    "finger_3": "Blues",
    "soft_pad": "Blues",
    "hand": "Blues", 
    "wrist": "Blues",
    "Front_PAD_corner": "Greens",
    "Back_PAD_corner": "Greens",
    "Lever_ball": "Reds",
    "Lever_base": "Reds",
    "Lever_corner": "Reds",
}

from src.BetaMouv.Trial import Trial

filenames = list(output_dir.glob("*.csv"))
n=0
for path in tqdm(filenames, desc="plotting"):
    
    trial = Trial(path.parent / f"{path.stem}.yaml")
    trial.load_yaml(path.parent / f"{path.stem}.yaml")
    # print({bp: stat["stage_outcomes"]["validation"]["reason"] for bp, stat in trial.trajectories.items()})

    df = pd.read_csv(path, header=[0, 1, 2], index_col=0)
    df = df.droplevel(0, axis=1)
    df = df.iloc[1:].reset_index(drop=True)
    df.index = np.arange(len(df)) / 125
    df.index.name = "t"

    plot_tracking(df, skeleton, delta_T=1, color_by_time=False)


plotting: 100%|██████████| 201/201 [01:32<00:00,  2.18it/s]


In [ ]:
traj_data = []

n=0
for trial in project.trialgroup.trials:
    if not trial.is_valid():
        continue

    for bp, stat in trial.trajectories.items():
        data = {info: val for  info, val in stat.items()}
        data["id"] = n
        data["trial_name"] = trial.name

        traj_data.append(data)


traj_data_df = pd.DataFrame(traj_data)
traj_data_df.to_csv("traj_data_df.csv")

## Success rate

In [ ]:
project.trialgroup.build_df(df_dir, "trial_success")
project.trialgroup.build_df(df_dir, "bodypart_success")

In [ ]:
project.trialgroup.trial_success_rate(u.make_path(analysis_dir, "trial_success_rate.png"))

In [ ]:
project.trialgroup.trajectories_success_rate(u.make_path(analysis_dir, "trajectories_success_rate.png"))

In [ ]:

project.trialgroup.sunburst_metadata(output_dir=u.make_dir(analysis_dir / "metadata_sunburst"),
                                  subfig_group="subject", 
                                  groups=["laser_type"],
                                  title=project.trialgroup.group_name)

In [ ]:
project.trialgroup.sunburst_metadata(output_dir=u.make_dir(analysis_dir / "metadata_sunburst"), 
                                  subfig_group="condition", 
                                  groups=["subject", "laser_type", "laser_intensity"],
                                  title=project.trialgroup.group_name)

In [ ]:
project.trialgroup.sunburst_metadata(output_dir=u.make_dir(analysis_dir / "metadata_sunburst"), 
                                  subfig_group=None, 
                                  groups=["subject", "is_valid", "failing_step"],
                                  title=project.trialgroup.group_name)



## Behavior analysis

In [ ]:
project.trialgroup.build_df(df_dir, "behavior")

In [ ]:
features = [
            # "relative_t",
            "x", 
            "y",
            "area", 
            "centroid_direction", 
            "centroid_velocity", 
            # "centroid_speed", 
            # "centroid_acc", 
            "f3_sp_distance", 
            "f3_sp_angle", 
            "centroid_lever_dist"
]

### PCA

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
labels = [
    "name", "date", "subject", "spatial_behavior", "t",
    "laser_type", "laser_state", "condition", "laser_intensity"
]
BEHAVIOR_PALETTE = {
    "reach":"yellow", 
    "open":"orange", 
    "grasp":"blue", 
    "press":"green", 
    "none":"gray"
}

In [ ]:
raw_df = project.trialgroup.clean_df(project.trialgroup._behavior_df)
raw_df = raw_df[features + labels].dropna()
label_df = raw_df[labels]
df = raw_df[features]
print(len(label_df), len(df))
print(raw_df.columns)
print(df.columns)

In [ ]:

# X is your DataFrame containing features
X = df.select_dtypes(include="number")

# 1. Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 3. Store results in a new DataFrame
pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=df.index
).join(label_df)

pca_df.head()

In [ ]:
count = pca_df.groupby("spatial_behavior").count()
g = sns.catplot(
    data=pca_df, kind="count",
    y="spatial_behavior",
    hue="spatial_behavior", palette=BEHAVIOR_PALETTE,
)

# add count on bars
for ax in g.axes.flat: 
    for container in ax.containers: 
        ax.bar_label(container, fmt="%d", padding=3)


In [ ]:
print(pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=["PC1", "PC2"]
)

loadings

In [ ]:
fig, ax = plt.subplots()

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="spatial_behavior",
    palette=BEHAVIOR_PALETTE, s=12,
    alpha=0.5, ax=ax, 
)

loadings = pca.components_.T 

scale = min(
    pca_df["PC1"].max() - pca_df["PC1"].min(), 
    pca_df["PC2"].max() - pca_df["PC2"].min()
) * 0.4

for i, feature in enumerate(features): 
    x_loading = loadings[i, 0] * scale
    y_loading = loadings[i, 1] * scale

    ax.annotate("", xy=(x_loading, y_loading), 
                xytext=(0, 0), 
                arrowprops=dict(arrowstyle="->", color="black", lw=1.2,))

    ax.text(x_loading*1.2, y_loading*1.2, feature, color="black", fontsize=8, ha="center", va="center")

ax.legend(loc="upper right", fontsize=8)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title(f"PCA by spatial behavior, n point = {len(pca_df)}")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

g = sns.FacetGrid(data=pca_df, 
                  col="laser_type", 
                  row="laser_intensity",
                  margin_titles=True,)

g.map_dataframe(
    sns.scatterplot,
    x="PC1",
    y="PC2",
    hue="laser_state", 
    palette="tab10",
    alpha=0.5, s=12
)
g.add_legend(loc="upper right")
g.set_axis_labels(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", 
                  f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
g.set_titles(row_template="{row_name}", col_template="{col_name}")
g.figure.suptitle(f"PCA by group - n point = {len(pca_df)}")
plt.tight_layout()
plt.show()

Below is a feature set organised by what it captures. It assumes a fixed camera, 2D tracking, and a lever/pad setup where the reach ends in a lever press or grasp. Adjust if your setup differs.

### 0. Preprocessing before any feature
- **Reference frame:** express hand points relative to the task geometry, not raw pixels. Use `Lever_base`→`Lever_corner` or the pad corners as the axes, so features are invariant to small camera or setup shifts.
- **Scale:** convert to mm using a known distance (pad length, lever length), or normalise by hand size (e.g. median `wrist`–`Hand_base` distance).
- **Smoothing:** apply Savitzky-Golay or a low-pass filter *before* differentiating. Derivatives amplify the noise left after interpolation.
- **Angles:** encode them as sin/cos, or unwrap them, so wrap-around at ±180° doesn't create fake jumps.

### 1. Hand transport (where the hand goes)
- x/y of `wrist`, `Hand_base`, `hand` (and the centroid of the hand points), relative to `Lever_ball` and the pad.
- Speed, acceleration, jerk, and velocity direction angle for `wrist` and `Hand_base`.
- Distance from the hand to `Lever_ball`, the lever line, `Front_PAD_corner` and `Back_PAD_corner`.
- Path-based, per-trial measures: path length, straightness (straight-line distance / path length), curvature, number of velocity peaks (sub-movements), peak speed and time to peak.

### 2. Hand orientation
- Angle of the `wrist`→`Hand_base` vector, as a pronation/supination or tilt proxy, plus its angular velocity.
- Angle of that vector relative to the lever axis.
- Angle of `Hand_base`→`soft_pad`, and the palm-side orientation.

### 3. Finger configuration (the grasp)
- **Joint angles** for each finger chain, e.g. `Hand_base`–`Base_finger_2`–`Phalanx_1_finger_2`, and `Base_finger_2`–`Phalanx_1_finger_2`–`finger_2`. Same for finger 3. These give flexion/extension.
- **Finger extension ratio:** distance(`Hand_base`, fingertip) / maximal finger length. This is a robust, scale-free measure.
- **Aperture:** distances `finger_1`–`finger_2`, `finger_1`–`finger_3`, and `finger_2`–`finger_3`.
- **Finger spread angle:** angle between `Hand_base`→`Base_finger_2` and `Hand_base`→`Base_finger_3`.
- **Hand area:** convex hull or polygon area over the hand points, a compact open/closed measure.
- Angular velocities of the above, so you can separate opening from closing.

### 4. Hand–object interaction
- Fingertip, `soft_pad` and `hand` distance to `Lever_ball` (minimum across fingers, and which finger is nearest).
- Lever state: lever angle (`Lever_base`→`Lever_ball` or `Lever_corner`), displacement, and velocity. This is your task outcome, so it is useful for labeling clusters.
- Contact proxies: distance below a threshold, plus the hand height relative to the pad plane (lift, touching, retracting).

### 5. Quality control features (don't cluster on them)
- Mean DLC likelihood per frame and the fraction of interpolated frames. Use these to check that clusters aren't driven by tracking artifacts.

### From features to clustering
- **Normalise and prune:** z-score or robust-scale, then drop highly correlated features. You'll end up with roughly 40–80 features, which is fine.
- **Choose a unit of analysis:**
  - *Per-frame or sliding-window* (e.g. 100–300 ms windows with mean, std, min and max of each feature): best for discovering sub-movements such as reach, pronate, grasp, retract. Follow with UMAP or PCA, then HDBSCAN or GMM.
  - *Per-trial summaries* (peak speed, duration, aperture at lever contact, etc.): best for comparing trials, sessions or animals.
  - *Time-warped trajectories* (DTW or fixed phase normalisation, then PCA or functional PCA): good for comparing reach shapes.
- **Align trials to an event** such as movement onset (speed threshold) or lever contact, and keep the phase of the trial (time since onset) as a feature or a colouring variable.
- **Cross-check:** compare your clusters against B-SOiD or Keypoint-MoSeq on the same keypoints. If the hand-crafted features and the unsupervised pose models agree, the labels are more trustworthy.

### Caveats
- With 2D tracking, angles are projections, so finger flexion can be confounded by hand rotation out of the image plane. Features that are most robust to this are distances normalised by hand size.
- Check which of your points are really independent. `hand`, `soft_pad` and `Hand_base` may be close together, so don't count them as three separate signals.

If you tell me the frame rate, whether the tracking is 2D or 3D, and what the pad and lever are used for (reward delivery, press threshold, etc.), I can narrow this to a shorter list and write the feature extraction code (pandas/numpy) for your DLC output format.

In [ ]:
for feat in features: 
    project.trialgroup.plot_tendency_beha_features(
                        save_as=u.make_path(analysis_dir / "behavior" / "with_behavior_span", f"{feat}.png"),
                        x="relative_t", y=feat, show_beha="spatial_behavior")

    project.trialgroup.plot_tendency_beha_features(
                        save_as=u.make_path(analysis_dir / "behavior", f"{feat}.png"),
                        x="relative_t", y=feat, show_beha=None)



In [ ]:
project.trialgroup.lineplot_traj_centroid(save_as=u.make_path(analysis_dir / "behavior", f"centroid_traj_2.png"),) 

Distribution of each features, per protocols

In [ ]:
for l_type in ["Beta","Conti"] : 
    for l_intensity in ["low", "high"]: 
        for l_state in ["LaserOn", "LaserOff"] : 
            project.trialgroup.distri_feat_per_behavior(behavior_varname="spatial_behavior", laser_type=l_type, laser_intensity=l_intensity, laser_state=l_state,
                                                    save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", f"spatial_behavior_{l_type}_{l_intensity}_{l_state}.svg"),
                                                    features_list=features)



Distribution of each features, for the whole dataset

In [ ]:
project.trialgroup.distri_feat_per_behavior(behavior_varname="spatial_behavior",
                                        save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", "spatial_behavior_state_all.svg"),
                                        features_list=features)


### HMM
Behavior exploration : HMM fitting

In [ ]:
df = project.trialgroup._behavior_df
Behavior.HMM_exploration(df, analysis_dir, load=None)


In [ ]:
project.trialgroup.distri_feat_per_behavior(behavior_varname="pred_state", 
                                        save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", "HMM_pred_state_all.svg"),
                                        features_list=features)


In [ ]:
for l_type in ["Beta","Conti"] : 
    for l_intensity in ["low", "high"]: 
        for l_state in ["LaserOn", "LaserOff"] : 
            project.trialgroup.distri_feat_per_behavior(behavior_varname="pred_state", laser_type=l_type, laser_intensity=l_intensity, laser_state=l_state,
                                                    save_as=u.make_path(analysis_dir / "behavior" / "distri_feat_per_behavior", f"HMM_pred_state_{l_type}_{l_intensity}_{l_state}.svg"),
                                                    features_list=features)


## Trajectory analysis

In [ ]:
project.trialgroup.build_df(df_dir, "kinematics")

In [ ]:
timeseries_metric = [
                    "x", 
                    "y", 
                    "instant_velocity", 
                    "instant_speed", 
                    "acc", 
                    "signed_acc", 
                    "lever_distance",
                    "angle"
                    ]

bodypart = "finger_3" 
project.trialgroup.remove_NOstim = True
project.trialgroup.crop_laser_period = True

In [ ]:
for val in timeseries_metric:
    project.trialgroup.plot_tendency_metrics(
        value=val,
        save_as=u.make_path(analysis_dir / "tendency" / bodypart, f"{val}.svg"),
        # show_units=True, 
        show_beha=None
    )

Lineplot traj with behavior boxes

In [ ]:
trial = project.trialgroup.trials[1]
behavior_obj = Behavior(trial.trajectories, trial.time_pad_off, trial.cm_per_pixel)
project.trialgroup.lineplot_traj(save_as=analysis_dir / f"{bodypart}_all_traj_with_beha_zones.svg", Beha=behavior_obj)


Lineplot all valid traj

In [ ]:
project.trialgroup.lineplot_traj(save_as=u.make_path(analysis_dir, f"{bodypart}_all_traj.svg"))

Lineplot all valid traj + facet grib per protocol

In [ ]:
project.trialgroup.lineplot_traj_per_indentity(save_as=analysis_dir / f"{bodypart}_traj_per_indentity.svg")

## Dstribution analysis (scalar metrics)

In [ ]:
project.trialgroup.build_df(df_dir, "scalar")

In [ ]:
bodypart = "finger_3"
metrics = [
    "avg_velocity_laser_period", 
    "tortuosity_laser_period", 
    "tortuosity_pad_to_lever", 
]

In [ ]:
comparisons = [
    # Conti vs Beta
    ("Conti_LaserOff.low",  "Beta_LaserOff.low"),
    ("Conti_LaserOff.high", "Beta_LaserOff.high"),
    ("Conti_LaserOn.low",   "Beta_LaserOn.low"),
    ("Conti_LaserOn.high",  "Beta_LaserOn.high"),

    # Off vs On
    ("Conti_LaserOff.low",  "Conti_LaserOn.low"),
    ("Conti_LaserOff.high", "Conti_LaserOn.high"),
    ("Beta_LaserOff.low",   "Beta_LaserOn.low"),
    ("Beta_LaserOff.high",  "Beta_LaserOn.high"),

    # low vs high
    ("Beta_LaserOff.low",   "Beta_LaserOff.high"),
    ("Beta_LaserOn.low",    "Beta_LaserOn.high"),
    ("Conti_LaserOff.low",  "Conti_LaserOff.high"),
    ("Conti_LaserOn.low",   "Conti_LaserOn.high"),
]

In [ ]:
for metric in metrics: 
    print(metric)
    project.trialgroup.violin_statistic(comparisons, metric=metric, 
                                     save_as=u.make_path(analysis_dir / "distribution" / bodypart, f"{metric}.svg"))

Laser Off merged

In [ ]:
comparisons_merge_Off = [
    # conti vs beta
    ("Conti.low",  "Beta.low"),
    ("Conti.high", "Beta.high"),

    # low vs high
    ("LaserOff.high", "Conti.high"),
    ("LaserOff.high",  "Beta.high"),
    ("LaserOff.low",    "Beta.low"),
    ("LaserOff.low",   "Conti.low"),

    # condition high vs condition low
    ("Conti.low", "Conti.high"),
    ("Beta.low",  "Beta.high"),

    # Beta high ON vs Conti low ON 
    # (because conti low is suppose to be as damagefull as the beta high)
    ("Beta.high", "Conti.low"),
    ("Beta.low", "Conti.high")
]

In [ ]:
for metric in metrics: 
    print(metric)
    project.trialgroup.violin_statistic(comparisons_merge_Off, metric=metric,  merge_Off=True,
                                     save_as=u.make_path(analysis_dir / "distribution" / bodypart, f"{metric}_merged_off.svg"),
                                     order=["LaserOff", "Conti", "Beta"])


## Single traj timeseries analysis

In [ ]:
for trial in tqdm(project.trialgroup.trials, desc="plotting timeseries"): 
    if not trial.is_valid(): 
        continue
    
    traj: TrajectoryDLC = trial.trajectories.get(bodypart)

    if not traj.is_valid(): 
        continue

    fig, ax = plt.subplots()

    ax = traj.plot_traj(traj.coords, ax, fig, color = "orange", label="raw")
    ax = traj.plot_traj(traj.interpolated_coords, ax, fig, color = "green", label="interpolated")
    ax = traj.plot_traj(traj.clean_coords, ax, fig, color = "blue", label="clean")


    name = trial.name
    title = name[:len(name)//2] + "\n" + name[len(name)//2:]

    ax.set(
        title=title,
        xlabel="X position (cm)",
        ylabel="Y position (cm)",
    )
    fig.savefig(u.make_path(analysis_dir / "traj_per_trial" / bodypart, f"{trial.name}.svg"))
    plt.close()

    for val in timeseries_metric: 

        traj.plot_tendency(value=val, time_pad_off=trial.time_pad_off,
                           laser_state=trial.laser_state,
                           group=trial.group,
                           show_angle= val == "angle",
                           save_as=u.make_path(analysis_dir / "tendency_per_trial" / bodypart / trial.name , f"{val}.svg"))


### DLCTracking animation

In [ ]:
# path = Path("/home/ninjayu/IMN_Kinematics/data/BetaMouv/raw/subject_#531/Rat_#531LeftHanded_20240705_ContiMT300_0,75mW_RightHemiCHR_L1L25050_C001H001S0002/"
#     "pred_results_Rat_#531LeftHanded_20240705_ContiMT300_0,75mW_RightHemiCHR_L1L25050_C001H001S0002_clip_01.csv")
# path = Path("/home/poemiti/IMN_Kinematics/data/BetaMouv/raw/subject_#531/Rat_#531LeftHanded_20240626_ContiMT300_RightHemiCHR_L1L25050_C001H001S0001/"
#             "pred_results_Rat_#531LeftHanded_20240626_ContiMT300_RightHemiCHR_L1L25050_C001H001S0001_clip_00.csv")
# clip_path = Path("/home/poemiti/IMN_Kinematics/data/BetaMouv/raw/subject_#531/Rat_#531LeftHanded_20240626_ContiMT300_RightHemiCHR_L1L25050_C001H001S0001/"
#             "Rat_#531LeftHanded_20240626_ContiMT300_RightHemiCHR_L1L25050_C001H001S0001_clip_00.mp4")
path = Path("/home/poemiti/IMN_Kinematics/data/BetaMouv/raw/subject_#531/Rat_#531LeftHanded_20240712_ContiMT300_0,75mW_RightHemiCHR_L1L25050_C001H001S0002/"
            "pred_results_Rat_#531LeftHanded_20240712_ContiMT300_0,75mW_RightHemiCHR_L1L25050_C001H001S0002_clip_15.csv")
clip_path = Path("/home/poemiti/IMN_Kinematics/data/BetaMouv/raw/subject_#531/Rat_#531LeftHanded_20240712_ContiMT300_0,75mW_RightHemiCHR_L1L25050_C001H001S0002/"
            "Rat_#531LeftHanded_20240712_ContiMT300_0,75mW_RightHemiCHR_L1L25050_C001H001S0002_clip_15.mp4")



df = pd.read_csv(path, header=[0, 1, 2], index_col=0)
df = df.droplevel(0, axis=1)
df = df.iloc[1:].reset_index(drop=True)
df.index = np.arange(len(df)) / 125
df.index.name = "t"
df

In [ ]:
df_long = (
    df.stack(level=0)
      .reset_index()
      .rename(columns={'level_1': 'bodyparts'})
      .rename_axis(None, axis=1)
)
df_long

In [ ]:
df_long["bodyparts"].unique()

### Display pose in time (plotly)

In [ ]:
skeleton=[
    ("Hand_base", "Base_finger_2"),
    ("Hand_base", "Base_finger_3"),
    ("Hand_base", "soft_pad"),
    ("Base_finger_2", "Phalanx_1_finger_2"),
    ("Base_finger_3", "Phalanx_1_finger_3"),
    ("Phalanx_1_finger_2", "finger_2"),
    ("Phalanx_1_finger_3", "finger_3"),
    ("Front_PAD_corner", "Back_PAD_corner"),
    ("Lever_base", "Lever_corner"),
]
bodyparts_to_use = [
    "Hand_base",
    "Base_finger_2",
    "Base_finger_3",
    "Phalanx_1_finger_2",
    "Phalanx_1_finger_3",
    # "finger_1",
    "finger_2",
    "finger_3",
    "soft_pad",
    "Front_PAD_corner",
    "Back_PAD_corner",
    "Lever_ball",
    "Lever_base",
    "Lever_corner",
]
color_map = {
    "Hand_base": "lightskyblue",
    "Base_finger_2": "lightskyblue",
    "Base_finger_3": "lightskyblue",
    "Phalanx_1_finger_2": "lightskyblue",
    "Phalanx_1_finger_3": "lightskyblue",
    "finger_1": "lightskyblue",
    "finger_2": "lightskyblue",
    "finger_3": "lightskyblue",
    "soft_pad": "lightskyblue",
    "Front_PAD_corner": "olive",
    "Back_PAD_corner": "olive",
    "Lever_ball": "tomato",
    "Lever_base": "tomato",
    "Lever_corner": "tomato",
}
color_scale_map = {
    "Hand_base": "Blues",
    "Base_finger_2": "Blues",
    "Base_finger_3": "Blues",
    "Phalanx_1_finger_2": "Blues",
    "Phalanx_1_finger_3": "Blues",
    "finger_1": "Blues",
    "finger_2": "Blues",
    "finger_3": "Blues",
    "soft_pad": "Blues",
    "Front_PAD_corner": "Greens",
    "Back_PAD_corner": "Greens",
    "Lever_ball": "Reds",
    "Lever_base": "Reds",
    "Lever_corner": "Reds",
}
times = sorted(df_long["t"].unique())
parts = sorted(bodyparts_to_use)
df_long = df_long[df_long["bodyparts"] != "wrist"]
df_long = df_long[df_long["bodyparts"] != "hand"]
df_long = df_long[df_long["bodyparts"] != "finger_1"]

# Pivot once for fast lookup: one row per (t, bodypart)
coords = df_long.set_index(["t", "bodyparts"])[["x", "y"]]

In [ ]:
import plotly.graph_objects as go

times = sorted(df_long["t"].unique())
parts = sorted(df_long["bodyparts"].unique())

# Pivot once for fast lookup: one row per (t, bodypart)
coords = df_long.set_index(["t", "bodyparts"])[["x", "y"]]

def frame_data(t):
    sub = coords.loc[t]  # index: bodyparts, columns: x, y

    # Points
    pts = go.Scatter(
        x=sub["x"], y=sub["y"],
        mode="markers",
        text=sub.index,
        hoverinfo="text",
        marker=dict(size=10, 
                    color=[color_map[p] for p in sub.index]),
        name="bodyparts",
    )

    # Skeleton: a single trace, bones separated by None
    xs, ys = [], []
    for a, b in skeleton:
        if a in sub.index and b in sub.index:
            xs += [sub.at[a, "x"], sub.at[b, "x"], None]
            ys += [sub.at[a, "y"], sub.at[b, "y"], None]
    bones = go.Scatter(
        x=xs, y=ys,
        mode="lines",
        line=dict(width=3, color="black"),
        hoverinfo="skip",
        name="skeleton",
    )
    return [bones, pts]  # bones first so points are drawn on top


fig = go.Figure(
    data=frame_data(times[0]),
    frames=[go.Frame(data=frame_data(t), name=str(t)) for t in times],
)

fig.update_layout(
    xaxis=dict(range=[0, 350], constrain="domain"),
    yaxis=dict(range=[400, 150], scaleanchor="x",),  # keep aspect ratio 1:1
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, dict(frame=dict(duration=50, redraw=True),
                                  fromcurrent=True)]),
            dict(label="Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")]),
        ],
    )],
    sliders=[dict(
        steps=[dict(method="animate",
                    args=[[str(t)], dict(mode="immediate",
                                         frame=dict(duration=0, redraw=True))],
                    label=str(t))
               for t in times],
    )], template="plotly_white",
)

fig.write_html(u.make_path(analysis_dir / "single_trial" / f"{path.stem}" , "skeleton_animation.html"), auto_open=True)

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_tracking(df, skeleton, indices=None, delta_T=10, by_label=False, color_by_time=True) :
    bodyparts = bodyparts_to_use

    if indices is None:
        sampled = df.iloc[::delta_T]
        title = f"Bodypart tracking (every {delta_T} frames)"
    else:
        sampled = df.loc[indices] if by_label else df.iloc[indices]
        title = f"Bodypart tracking ({len(sampled)} selected frames)"
    
    n = len(sampled)
    t_norm = np.linspace(0.1, 1.0, n)  # 0.3 -> skip the near-white end of the scale
    colors = px.colors.qualitative.Plotly

    fig = go.Figure()

    # --- Skeleton ---
    if color_by_time:
        # One trace per sampled frame, colored along a sequential scale
        n = len(sampled)
        frame_colors = px.colors.sample_colorscale(
            "Blues", [i / max(n - 1, 1) for i in range(n)])

        for k, (idx, row) in enumerate(sampled.iterrows()):
            xs, ys = [], []
            for a, b in skeleton:
                xs += [row[a]["x"], row[b]["x"], None]
                ys += [row[a]["y"], row[b]["y"], None]
            fig.add_trace(go.Scatter(
                x=xs, y=ys, mode="lines",
                line=dict(width=1.5, color=frame_colors),
                opacity=0.8,
                hoverinfo="skip",
                name=f"t={idx}",
                legendgroup="skeleton",
                showlegend=False,
            ))

    else:
        for a, b in skeleton:
            seg_colors = px.colors.sample_colorscale(color_scale_map[a], t_norm)

            xa, xb = sampled[a]["x"].to_numpy(), sampled[b]["x"].to_numpy()
            ya, yb = sampled[a]["y"].to_numpy(), sampled[b]["y"].to_numpy()

            for k in range(n):
                fig.add_trace(go.Scatter(
                    x=[xa[k], xb[k]],
                    y=[ya[k], yb[k]],
                    mode="lines",
                    line=dict(width=5, color=seg_colors[k]),
                    opacity=1,
                    hoverinfo="skip",
                    showlegend=False,
                ))

    # --- Points (drawn after, so they sit on top) ---
    for i, bp in enumerate(bodyparts):
        point_colors = px.colors.sample_colorscale(color_scale_map[bp], t_norm)
        fig.add_trace(go.Scatter(
            x=sampled[bp]["x"], y=sampled[bp]["y"],
            mode="markers",
            name=bp,
            marker=dict(size=10, color=point_colors, opacity=0.9,
                        line=dict(width=0.5, color="black")),
        ))

    fig.update_layout(
        title=title,
        xaxis_title="X", yaxis_title="Y",
        template="plotly_white", width=900, height=800,
    )
    fig.update_yaxes(autorange="reversed", scaleanchor="x")
    fig.write_html(u.make_path(analysis_dir / "single_trial" / f"{path.stem}" , title), auto_open=True)
    
    return fig

In [ ]:
fig = plot_tracking(df, skeleton, indices=[0.416, 1.048, 1.088, 1.128, 1.16,
                                           1.176, 1.192, 1.216, 1.232, 1.368], by_label=True, color_by_time=False)

In [ ]:
fig = plot_tracking(df, skeleton, indices=[0.224, 0.4, 0.448, 0.472, 0.52, 
                                           0.544, 0.592, 0.84], by_label=True, color_by_time=False)

### Video annotation

In [ ]:
# see same thing  with video
from src.BetaMouv.Video import Video

vid = Video(clip_path)
vid.annotate_video(output_path=u.make_path(analysis_dir / "annotated_video", f"{vid.name}.mp4"), 
                   bodyparts=bodyparts_to_use, csv_path=path, skeleton_path=Path("./skeleton.yaml"))